## Importação de bibliotecas e configuração global

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn import model_selection, preprocessing, decomposition, linear_model, metrics

plt.style.use("seaborn-v0_8")
plt.rcParams["font.size"] = 11
plt.rcParams["figure.figsize"] = (7, 5)

RANDOM_STATE = 42

BASE = "."
DIR_TABELAS = os.path.join(BASE, "TABELAS")
DIR_FIGURAS = os.path.join(BASE, "FIGURAS")
PATH_SERIES_MATRIX = os.path.join(BASE, "GSE124208_series_matrix.txt")
PATH_RAW_COUNTS = os.path.join(BASE, "GSE124208_raw_counts_GRCh38.p13_NCBI.tsv")
PATH_ANNOT = os.path.join(BASE, "Human.GRCh38.p13.annot.tsv")

os.makedirs(DIR_TABELAS, exist_ok=True)
os.makedirs(DIR_FIGURAS, exist_ok=True)


## Leitura dos metadados e definição do vetor Y (rótulos)

In [2]:
_path = PATH_SERIES_MATRIX
if not os.path.isfile(_path):
    _path = os.path.join(BASE, "SINDROME DE COCKAINE GSE124208", "GSE124208_series_matrix.txt")
geo = {}
with open(_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n\r")
        if not line.startswith("!"):
            continue
        parts = line.split("\t")
        key = parts[0].lstrip("!")
        vals = [p.strip('"') for p in parts[1:] if p.strip()]
        if key == "Sample_characteristics_ch1" and vals and "cell type:" in vals[0]:
            geo["Sample_cell_type"] = [v.replace("cell type: ", "").strip() for v in vals]
        elif key not in geo:
            geo[key] = vals

sample_id = geo["Sample_geo_accession"]
label_raw = geo["Sample_title"]
cell_type = geo.get("Sample_cell_type")
if not cell_type or len(cell_type) != len(sample_id):
    cell_type = ["iPSC" if "iPSC" in t else "MSC" if "MSC" in t else "NSC" if "NSC" in t else "" for t in label_raw]
condition = ["Ctrl" if "Ctrl" in t else "UV" if "UV" in t else "RS" if "RS" in t else "basal" for t in label_raw]
status = ["mut" if "mut" in t else "GC" for t in label_raw]

metadados = pd.DataFrame({
    "sample_id": sample_id,
    "label_raw": label_raw,
    "status": status,
    "cell_type": cell_type,
    "condition": condition,
})

Y = (metadados["status"] == "mut").astype(int).values
metadados.to_csv(os.path.join(DIR_TABELAS, "metadados_amostras.csv"), index=False)


## Leitura da matriz de contagens e alinhamento com metadados

In [3]:
_path_counts = PATH_RAW_COUNTS
if not os.path.isfile(_path_counts):
    _path_counts = os.path.join(BASE, "SINDROME DE COCKAINE GSE124208", "GSE124208_raw_counts_GRCh38.p13_NCBI.tsv")

contagens = pd.read_csv(_path_counts, sep="\t", index_col=0)
col_amostras = [c for c in contagens.columns if c in metadados["sample_id"].values]
ids_metadados = set(metadados["sample_id"])
ids_contagens = set(contagens.columns)
comum = ids_metadados & ids_contagens
so_metadados = ids_metadados - ids_contagens
so_contagens = ids_contagens - ids_metadados

conferencia = pd.DataFrame({
    "tipo": ["em_comum", "so_metadados", "so_contagens"],
    "ids": [",".join(sorted(comum)), ",".join(sorted(so_metadados)), ",".join(sorted(so_contagens))],
    "n": [len(comum), len(so_metadados), len(so_contagens)],
})
conferencia.to_csv(os.path.join(DIR_TABELAS, "conferencia_amostras_contagens.csv"), index=False)

ordem_amostras = list(metadados["sample_id"])
colunas_alinhadas = [c for c in ordem_amostras if c in contagens.columns]
X = contagens[colunas_alinhadas].copy()
metadados = metadados.set_index("sample_id").loc[colunas_alinhadas].reset_index()
Y = (metadados["status"] == "mut").astype(int).values

resumo = pd.DataFrame({
    "sample_id": colunas_alinhadas,
    "soma_contagens": X.sum(axis=0).values,
    "genes_nao_zero": (X > 0).sum(axis=0).values,
})
resumo.to_csv(os.path.join(DIR_TABELAS, "contagens_resumo.csv"), index=False)


## Filtro de genes (apenas expressão, fora da validação cruzada)

In [4]:
MIN_CONTAGEM = 1
MIN_AMOSTRAS = 2

n_genes_antes = X.shape[0]
mask = (X >= MIN_CONTAGEM).sum(axis=1) >= MIN_AMOSTRAS
X = X.loc[mask]
n_genes_depois = X.shape[0]

filtro_resumo = pd.DataFrame({
    "etapa": ["antes_filtro", "depois_filtro"],
    "n_genes": [n_genes_antes, n_genes_depois],
})
filtro_resumo.to_csv(os.path.join(DIR_TABELAS, "filtro_genes_resumo.csv"), index=False)
pd.DataFrame({"gene_id": X.index.tolist()}).to_csv(os.path.join(DIR_TABELAS, "genes_filtrados.csv"), index=False)


## Transformação log2(CPM+1) (entrada do pipeline)

In [5]:
totais = X.sum(axis=0)
LIMIAR_BAIXO = 1000
library_zero = (totais == 0).sum()
library_baixa = (totais > 0) & (totais < LIMIAR_BAIXO)
n_baixa = library_baixa.sum()
totais_safe = totais.replace(0, 1)

CPM = X / totais_safe * 1e6
X_log = np.log2(CPM + 1)

resumo_antes = X.values.flatten()
resumo_depois = X_log.values.flatten()
normalizacao_resumo = pd.DataFrame({
    "estatistica": ["min", "max", "media", "mediana", "n_amostras_library_zero", "n_amostras_library_baixa"],
    "antes_contagens": [
        float(resumo_antes.min()), float(resumo_antes.max()), float(resumo_antes.mean()), float(np.median(resumo_antes)), int(library_zero), int(n_baixa),
    ],
    "depois_log2cpm1": [
        float(resumo_depois.min()), float(resumo_depois.max()), float(resumo_depois.mean()), float(np.median(resumo_depois)), np.nan, np.nan,
    ],
})
normalizacao_resumo.to_csv(os.path.join(DIR_TABELAS, "normalizacao_resumo.csv"), index=False)


## PCA exploratório (apenas para inspeção e visualização)

In [6]:
N_COMP_EXPLORATORIO = 8
pca_exp = decomposition.PCA(n_components=N_COMP_EXPLORATORIO)
scores_exp = pca_exp.fit_transform(X_log.T)
loadings_exp = pca_exp.components_.T

scores_df = pd.DataFrame(scores_exp, columns=[f"PC{i+1}" for i in range(scores_exp.shape[1])])
scores_df.insert(0, "sample_id", metadados["sample_id"].values)
scores_df["status"] = metadados["status"].values
scores_df["cell_type"] = metadados["cell_type"].values
scores_df["condition"] = metadados["condition"].values
scores_df.to_csv(os.path.join(DIR_TABELAS, "X_pca_exploratorio.csv"), index=False)

loadings_df = pd.DataFrame(loadings_exp, index=X_log.index, columns=[f"PC{i+1}" for i in range(loadings_exp.shape[1])])
loadings_df.to_csv(os.path.join(DIR_TABELAS, "pca_loadings_exploratorio.csv"))

fig, ax = plt.subplots()
mut = metadados["status"] == "mut"
ax.scatter(scores_exp[mut, 0], scores_exp[mut, 1], c="C1", label="mut")
ax.scatter(scores_exp[~mut, 0], scores_exp[~mut, 1], c="C0", label="GC")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA exploratório (mut vs GC)")
ax.legend()
fig.savefig(os.path.join(DIR_FIGURAS, "pca_mut_vs_gc.png"), bbox_inches="tight")
plt.close(fig)


## Definição da estratégia de validação cruzada

In [7]:
N_FOLDS = 5
cv = model_selection.StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_alternativo = model_selection.StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=RANDOM_STATE)

OBS_VALIDACAO_GRUPOS = (
    "Validação por grupos biológicos (ex.: deixar de fora pares tipo celular/condição) é conceitualmente "
    "preferível para este estudo, mas pode ficar limitada pelo N pequeno e pelas combinações incompletas "
    "do GSE124208 (iPSC, MSC, NSC e Ctrl/UV/RS); documentar como possível extensão."
)


## Pipeline e validação cruzada (sem leakage)

In [8]:
from sklearn.base import clone

N_COMP_MODEL = 5

pipeline_modelo = Pipeline(
    steps=[
        ("scaler", preprocessing.StandardScaler()),
        ("pca", decomposition.PCA(n_components=N_COMP_MODEL)),
        (
            "logreg",
            linear_model.LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=1000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

resultados = []

X_array = X_log.T.to_numpy()

for i, (idx_treino, idx_teste) in enumerate(cv.split(X_array, Y), start=1):
    X_tr, X_te = X_array[idx_treino], X_array[idx_teste]
    y_tr, y_te = Y[idx_treino], Y[idx_teste]

    modelo = clone(pipeline_modelo)
    modelo.fit(X_tr, y_tr)

    y_pred = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]

    tn, fp, fn, tp = metrics.confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()

    acc = metrics.accuracy_score(y_te, y_pred)
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    esp = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    try:
        auc = metrics.roc_auc_score(y_te, y_proba)
    except ValueError:
        auc = np.nan

    resultados.append(
        {
            "fold": i,
            "accuracy": acc,
            "sensibilidade": sens,
            "especificidade": esp,
            "auc": auc,
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
    )

df_resultados = pd.DataFrame(resultados)
media = df_resultados.drop(columns=["fold", "tn", "fp", "fn", "tp"]).mean(numeric_only=True)
linha_media = {"fold": "media", "tn": np.nan, "fp": np.nan, "fn": np.nan, "tp": np.nan}
linha_media.update(media.to_dict())
df_resultados = pd.concat([df_resultados, pd.DataFrame([linha_media])], ignore_index=True)

df_resultados.to_csv(os.path.join(DIR_TABELAS, "resultados_cv_regressao_logistica.csv"), index=False)

## Avaliação final: matriz de confusão e curva ROC

In [9]:
y_proba_oof = model_selection.cross_val_predict(
    pipeline_modelo, X_log.T, Y, cv=cv, method="predict_proba"
)
y_pred_global = (y_proba_oof[:, 1] >= 0.5).astype(int)
cm_global = metrics.confusion_matrix(Y, y_pred_global, labels=[0, 1])
auc_global_oof = metrics.roc_auc_score(Y, y_proba_oof[:, 1])

linha_auc_global = pd.DataFrame([
    {"fold": "auc_global_oof", "accuracy": np.nan, "sensibilidade": np.nan, "especificidade": np.nan, "auc": auc_global_oof, "tn": np.nan, "fp": np.nan, "fn": np.nan, "tp": np.nan}
])
df_resultados = pd.concat([df_resultados, linha_auc_global], ignore_index=True)
df_resultados.to_csv(os.path.join(DIR_TABELAS, "resultados_cv_regressao_logistica.csv"), index=False)

fig, ax = plt.subplots()
sns.heatmap(cm_global, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["GC", "mut"], yticklabels=["GC", "mut"])
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusão global (regressão logística)")
fig.savefig(os.path.join(DIR_FIGURAS, "matriz_confusao_regressao_logistica.png"), bbox_inches="tight")
plt.close(fig)

fpr, tpr, _ = metrics.roc_curve(Y, y_proba_oof[:, 1])
fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f"ROC (AUC = {auc_global_oof:.3f})")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("Taxa de falsos positivos")
ax.set_ylabel("Taxa de verdadeiros positivos")
ax.set_title("Curva ROC (out-of-fold)")
ax.legend()
fig.savefig(os.path.join(DIR_FIGURAS, "roc_regressao_logistica.png"), bbox_inches="tight")
plt.close(fig)


## Interpretação dos coeficientes do modelo

In [10]:
pipeline_modelo.fit(X_log.T, Y)
logreg = pipeline_modelo.named_steps["logreg"]
coef_pcs = logreg.coef_[0]
pca_pipe = pipeline_modelo.named_steps["pca"]

coef_df = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(coef_pcs))],
    "coeficiente": coef_pcs,
})
coef_df["nota"] = "Coeficientes para interpretação; avaliação de desempenho é a da validação cruzada (Células 8–9)."
coef_df.to_csv(os.path.join(DIR_TABELAS, "coeficientes_regressao_logistica_pcs.csv"), index=False)

loadings_pipe = pca_pipe.components_.T
importancia_aprox = np.abs(loadings_pipe @ coef_pcs)
imp_df = pd.DataFrame({
    "gene_id": X_log.index,
    "importancia_aprox": importancia_aprox,
})
imp_df = imp_df.sort_values("importancia_aprox", ascending=False)
imp_df["nota"] = "Aproximação interpretativa a partir dos loadings do PCA; não é coeficiente do modelo no espaço gênico original."
imp_df.to_csv(os.path.join(DIR_TABELAS, "importancia_genes_modelo.csv"), index=False)


## Organização final das saídas e checklist

In [11]:
arquivos_tabelas = [
    "metadados_amostras.csv",
    "conferencia_amostras_contagens.csv",
    "filtro_genes_resumo.csv",
    "genes_filtrados.csv",
    "normalizacao_resumo.csv",
    "X_pca_exploratorio.csv",
    "pca_loadings_exploratorio.csv",
    "resultados_cv_regressao_logistica.csv",
    "coeficientes_regressao_logistica_pcs.csv",
    "importancia_genes_modelo.csv",
]
arquivos_figuras = [
    "pca_mut_vs_gc.png",
    "matriz_confusao_regressao_logistica.png",
    "roc_regressao_logistica.png",
]
checklist = []
for f in arquivos_tabelas:
    p = os.path.join(DIR_TABELAS, f)
    checklist.append({"tipo": "TABELAS", "arquivo": f, "existe": os.path.isfile(p)})
for f in arquivos_figuras:
    p = os.path.join(DIR_FIGURAS, f)
    checklist.append({"tipo": "FIGURAS", "arquivo": f, "existe": os.path.isfile(p)})
pd.DataFrame(checklist).to_csv(os.path.join(DIR_TABELAS, "checklist_saidas.csv"), index=False)

resumo_linhas = [
    "Parâmetros do pipeline:",
    f"  - Número de PCs: {N_COMP_MODEL} (decisão pragmática).",
    "  - Penalização: L2.",
    f"  - Folds (validação cruzada): {N_FOLDS} estratificados.",
    "  - class_weight: não utilizado (None).",
    "Limitações:",
    "  - Estrutura experimental do GSE124208 (tipos celulares e condições misturados).",
    "  - N pequeno (24 amostras); estimativas de desempenho podem ser instáveis.",
    "Extensão natural: comparar 3, 5 e 8 PCs por validação cruzada (ex.: busca de hiperparâmetro no pipeline).",
]
with open(os.path.join(DIR_TABELAS, "resumo_parametros_limites.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(resumo_linhas))
